In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.3 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import urllib.request
import zipfile
import os

/home/venv/trch/lib/python3.12/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._show_config()
/home/venv/trch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
print("Downloading MovieLens 100K dataset...")
urllib.request.urlretrieve(url, "ml-100k.zip")
with zipfile.ZipFile("ml-100k.zip", 'r') as zip_ref:
   zip_ref.extractall(".")
# Load and preprocess data
df = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
df['user_id'] = pd.Categorical(df['user_id']).codes
df['item_id'] = pd.Categorical(df['item_id']).codes
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
print(f"Dataset stats: {len(df)} interactions, {num_users} users, {num_items} items")

Dataset stats: 100000 interactions, 943 users, 1682 items


In [4]:
# Matrix Factorization Model using PyTorch Geometric
class MatrixFactorization(MessagePassing):
    def __init__(self, num_users, num_items, embedding_dim=50):
        super().__init__(aggr='add')  # Not used in this implementation but required
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        # Initialize embeddings
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_ids, item_ids):
        # Get embeddings and biases
        user_embed = self.user_embedding(user_ids)
        item_embed = self.item_embedding(item_ids)
        user_b = self.user_bias(user_ids).squeeze()
        item_b = self.item_bias(item_ids).squeeze()

        # Compute prediction: dot product + biases
        interaction = (user_embed * item_embed).sum(dim=1)
        prediction = interaction + user_b + item_b + self.global_bias
        return prediction

In [5]:
# Training function
def train(model, train_data, val_data, epochs=50, lr=0.01, weight_decay=1e-6):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_users = torch.tensor(train_data['user_id'].values, dtype=torch.long)
    train_items = torch.tensor(train_data['item_id'].values, dtype=torch.long)
    train_ratings = torch.tensor(train_data['rating'].values, dtype=torch.float32)

    val_users = torch.tensor(val_data['user_id'].values, dtype=torch.long)
    val_items = torch.tensor(val_data['item_id'].values, dtype=torch.long)
    val_ratings = torch.tensor(val_data['rating'].values, dtype=torch.float32)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        predictions = model(train_users, train_items)
        loss = criterion(predictions, train_ratings)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(val_users, val_items)
            val_rmse = torch.sqrt(nn.MSELoss()(val_pred, val_ratings))
        if epoch % 10 == 0:
            print(f'Epoch {epoch:3d} | Train Loss: {loss.item():.4f} | Val RMSE: {val_rmse:.4f}')
    return model

In [6]:
# Split data (80% train, 20% validation)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['user_id'])

# Initialize model
model = MatrixFactorization(num_users, num_items, embedding_dim=32)
trained_model = train(model, train_df, val_df, epochs=50, lr=0.01)

trained_model.eval()
test_users = torch.tensor(val_df['user_id'].values, dtype=torch.long)
test_items = torch.tensor(val_df['item_id'].values, dtype=torch.long)
test_ratings = torch.tensor(val_df['rating'].values, dtype=torch.float32)

with torch.no_grad():
 final_pred = trained_model(test_users, test_items)
 final_rmse = np.sqrt(mean_squared_error(test_ratings.numpy(), final_pred.numpy()))

print(f"\nFinal Test RMSE: {final_rmse:.4f}")

Epoch   0 | Train Loss: 13.7066 | Val RMSE: 3.6874
Epoch  10 | Train Loss: 10.6890 | Val RMSE: 3.2208
Epoch  20 | Train Loss: 5.2805 | Val RMSE: 2.1971
Epoch  30 | Train Loss: 1.1711 | Val RMSE: 1.0724
Epoch  40 | Train Loss: 1.2493 | Val RMSE: 1.1283

Final Test RMSE: 0.9623


In [27]:
preds=[]
userid=0
for itemid in range(1000):
 preds.append([userid, itemid, trained_model(torch.tensor([userid]), torch.tensor([itemid])).item()])
preds=np.array(preds)
idx=preds[:,2].argsort()
print(preds[idx][:10])

[[0.00000000e+00 8.56000000e+02 6.15032613e-01]
 [0.00000000e+00 8.13000000e+02 6.15365863e-01]
 [0.00000000e+00 8.57000000e+02 8.40131044e-01]
 [0.00000000e+00 8.51000000e+02 9.09790099e-01]
 [0.00000000e+00 8.96000000e+02 9.19810414e-01]
 [0.00000000e+00 8.29000000e+02 9.23142552e-01]
 [0.00000000e+00 7.83000000e+02 9.31985795e-01]
 [0.00000000e+00 5.98000000e+02 9.66036379e-01]
 [0.00000000e+00 9.09000000e+02 9.92568254e-01]
 [0.00000000e+00 4.38000000e+02 1.00608468e+00]]
